# Step 5 Part A: Build the training dataset for the deep hedging LSTM

Each episode (one option, one sampled day) becomes a padded sequence of features. We reuse the exact same option/hedge P&L logic validated in step 3, just reorganized into tensors.

In [ ]:
import pandas as pd
import numpy as np
import torch
from scipy.stats import norm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

train = pd.read_csv("btc_options_train.csv")
train["hour_bucket"] = pd.to_datetime(train["hour_bucket"])
train["sample_date"] = train["hour_bucket"].dt.date
train["T_years"] = train["time_to_maturity_days"] / 365
train["iv_decimal"] = train["mark_iv"] / 100
train["option_mid_usd"] = train["mid_price"] * train["underlying_price"]

def bs_delta(S, K, T, sigma, option_type, r=0.0):
    valid = (T > 0) & (sigma > 0)
    d1 = np.where(valid, (np.log(S / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(np.where(valid, T, 1))), 0)
    call_delta = norm.cdf(d1)
    put_delta = call_delta - 1.0
    is_call = (option_type == "call")
    return np.where(valid, np.where(is_call, call_delta, put_delta), 0.0)

train["bs_delta"] = bs_delta(train["underlying_price"].values, train["strike_price"].values,
                                train["T_years"].values, train["iv_decimal"].values, train["type"].values)
print(f"{len(train)} rows loaded, {train.groupby(['symbol','sample_date']).ngroups} episodes")

## Build padded episode tensors

Features per step: normalized moneyness (S/K, centered near 1), time-to-maturity (in years), implied vol, BS delta (a useful reference signal), and option type as a binary flag. We also keep spot price, option price, and cost rate separately since those feed the P&L calculation, not the network's input directly.

In [ ]:
MAX_LEN = 24
MIN_LEN = 4
BTC_TRANSACTION_COST_RATE = 0.0005

def build_episode_arrays(df, max_len=MAX_LEN, min_len=MIN_LEN):
    features_list, spot_list, option_pnl_list, mask_list = [], [], [], []
    meta = []

    for (symbol, sample_date), group in df.groupby(["symbol", "sample_date"]):
        group = group.sort_values("hour_bucket").reset_index(drop=True)
        n = len(group)
        if n < min_len:
            continue
        n_use = min(n, max_len)
        group = group.iloc[:n_use]

        moneyness = (group["underlying_price"] / group["strike_price"]).values
        ttm = group["T_years"].values
        iv = group["iv_decimal"].clip(upper=3.0).values  # cap extreme IV outliers at 300%
        delta = group["bs_delta"].values
        is_call = (group["type"] == "call").astype(float).values

        feat = np.stack([moneyness, ttm, iv, delta, is_call], axis=1)  # (n_use, 5)
        spot = group["underlying_price"].values
        option_mid_usd = group["option_mid_usd"].values
        option_pnl = -np.diff(option_mid_usd, prepend=option_mid_usd[0])
        option_pnl[0] = 0.0

        # pad to max_len
        pad_n = max_len - n_use
        if pad_n > 0:
            feat = np.vstack([feat, np.zeros((pad_n, feat.shape[1]))])
            spot = np.concatenate([spot, np.full(pad_n, spot[-1])])  # repeat last spot, harmless since masked
            option_pnl = np.concatenate([option_pnl, np.zeros(pad_n)])
        mask = np.array([1.0] * n_use + [0.0] * pad_n)

        features_list.append(feat)
        spot_list.append(spot)
        option_pnl_list.append(option_pnl)
        mask_list.append(mask)
        meta.append({"symbol": symbol, "sample_date": sample_date, "n_steps": n_use})

    features = np.stack(features_list)   # (n_episodes, max_len, 5)
    spots = np.stack(spot_list)          # (n_episodes, max_len)
    option_pnls = np.stack(option_pnl_list)
    masks = np.stack(mask_list)
    return features, spots, option_pnls, masks, pd.DataFrame(meta)

train_features, train_spots, train_option_pnls, train_masks, train_meta = build_episode_arrays(train)
print(f"Built {train_features.shape[0]} episodes, feature shape {train_features.shape}")
print(f"Feature ranges (moneyness, ttm, iv, delta, is_call):")
for i, name in enumerate(["moneyness", "ttm", "iv", "delta", "is_call"]):
    valid_vals = train_features[:, :, i][train_masks == 1]
    print(f"  {name}: min={valid_vals.min():.4f}, max={valid_vals.max():.4f}, mean={valid_vals.mean():.4f}")

## Normalize features

Neural nets train better when inputs are roughly similar scale. moneyness/delta are already ~[-1,2]ish, but ttm (years, e.g. 0.01-1.0) and iv (0.3-1.5) benefit from standardization. We fit normalization stats on TRAIN ONLY, then apply the same stats to val/test later -- this avoids leaking test statistics into training, a classic mistake to avoid.

In [ ]:
# Compute mean/std per feature using only valid (masked) entries
feature_names = ["moneyness", "ttm", "iv", "delta", "is_call"]
norm_stats = {}
for i, name in enumerate(feature_names):
    valid_vals = train_features[:, :, i][train_masks == 1]
    norm_stats[name] = {"mean": valid_vals.mean(), "std": valid_vals.std() + 1e-8}

def normalize_features(features, stats):
    normed = features.copy()
    for i, name in enumerate(feature_names):
        normed[:, :, i] = (features[:, :, i] - stats[name]["mean"]) / stats[name]["std"]
    return normed

train_features_normed = normalize_features(train_features, norm_stats)
print("Normalization stats (from TRAIN, to be reused for val/test):")
for name, s in norm_stats.items():
    print(f"  {name}: mean={s['mean']:.4f}, std={s['std']:.4f}")

# Save everything needed to reload this later without recomputing
np.savez("train_episode_tensors.npz",
         features=train_features_normed, spots=train_spots,
         option_pnls=train_option_pnls, masks=train_masks)
train_meta.to_csv("train_episode_meta.csv", index=False)
import json
json.dump({k: {kk: float(vv) for kk, vv in v.items()} for k, v in norm_stats.items()}, open("norm_stats.json", "w"))
print("\nSaved train_episode_tensors.npz, train_episode_meta.csv, norm_stats.json")